# Biohub Strategy EDA and Validation Set

Bu notebook, temizlenen 199-pair dataset uzerinde stratejiye donuk EDA yapar: GT yogunluk, division listesi, representative intensity, validation sample secimi ve runtime profiling iskeleti.

Amac: submission notebook'a gecmeden once neyi optimize edecegimizi olcmek.

## 1. Runtime ve Importlar

In [ ]:
from pathlib import Path
import json
import math
import os
import subprocess
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print('Not running in Colab or Drive already unavailable.')

try:
    import zarr  # noqa: F401
except Exception:
    print('Installing missing zarr dependencies...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'zarr', 'numcodecs'])

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)

## 2. Proje ve Dataset Bul

In [ ]:
PROJECT_CANDIDATES = [
    Path('/content/drive/MyDrive/Biohub - Cell Tracking During Development'),
    Path.cwd(),
]

PROJECT_DIR = None
for p in PROJECT_CANDIDATES:
    if (p / 'biohub_baseline.py').exists():
        PROJECT_DIR = p
        break

if PROJECT_DIR is None:
    raise FileNotFoundError('biohub_baseline.py bulunamadi. Notebook proje klasorunden veya Drive proje klasorunden calismali.')

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import biohub_baseline as bh


def find_dataset_base():
    candidates = [
        PROJECT_DIR / 'data',
        PROJECT_DIR / 'data' / 'biohub-cell-tracking-during-development',
        Path('/kaggle/input/competitions/biohub-cell-tracking-during-development'),
        Path('/kaggle/input/biohub-cell-tracking-during-development'),
    ]
    for c in candidates:
        if (c / 'train').exists() and (c / 'test').exists():
            return c
    raise FileNotFoundError('train/test iceren dataset base bulunamadi.')


BASE_DIR = find_dataset_base()
TRAIN_DIR = BASE_DIR / 'train'
TEST_DIR = BASE_DIR / 'test'
OUTPUT_DIR = PROJECT_DIR / 'reports'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_DIR:', PROJECT_DIR)
print('BASE_DIR:', BASE_DIR)
print('REPORTS_DIR:', OUTPUT_DIR)

## 3. Ayarlar

In [ ]:
EXPECTED_TRAIN_PAIRS = 199
EXPECTED_TEST_ZARR = 4

RUN_FULL_GEFF_SUMMARY = True
FORCE_REBUILD_GEFF_SUMMARY = False

RUN_REPRESENTATIVE_INTENSITY = True
INTENSITY_TIMEPOINTS = [0, 24, 49, 74, 99]

RUN_SELECTED_BOUNDARY_ANALYSIS = True

# Runtime profiling detection calistirir; simdilik kapali tut.
RUN_RUNTIME_PROFILE = False
RUNTIME_PROFILE_TIMEPOINTS = 5

SCALES = np.array([1.625, 0.40625, 0.40625], dtype=float)
print('settings ready')

## 4. Pair ve Zarr Kontrolu

In [ ]:
train_zarr_paths = sorted(TRAIN_DIR.glob('*.zarr'))
train_geff_paths = sorted(TRAIN_DIR.glob('*.geff'))
test_zarr_paths = sorted(TEST_DIR.glob('*.zarr'))

zarr_ids = {p.stem for p in train_zarr_paths}
geff_ids = {p.stem for p in train_geff_paths}
paired_ids = sorted(zarr_ids & geff_ids)

print('train zarr:', len(zarr_ids))
print('train geff:', len(geff_ids))
print('paired:', len(paired_ids))
print('test zarr:', len(test_zarr_paths))
print('geff without zarr:', sorted(geff_ids - zarr_ids))
print('zarr without geff:', sorted(zarr_ids - geff_ids))

assert len(paired_ids) == EXPECTED_TRAIN_PAIRS, 'Expected 199 paired train samples.'
assert len(test_zarr_paths) == EXPECTED_TEST_ZARR, 'Expected 4 public test zarr samples.'

rows = []
bad = []
for p in tqdm(train_zarr_paths, desc='zarr metadata train'):
    try:
        img = bh.open_image_zarr(p, print_info=False)
        rows.append({
            'split': 'train', 'sample_id': p.stem, 'embryo_id': p.stem.split('_')[0],
            'T': int(img.shape[0]), 'Z': int(img.shape[1]), 'Y': int(img.shape[2]), 'X': int(img.shape[3]),
            'dtype': str(img.dtype), 'chunks': tuple(int(x) for x in getattr(img, 'chunks', ())),
            'error': np.nan,
        })
    except Exception as exc:
        bad.append((p.stem, repr(exc)))
        rows.append({'split': 'train', 'sample_id': p.stem, 'embryo_id': p.stem.split('_')[0], 'error': repr(exc)})

zarr_meta_df = pd.DataFrame(rows)
display(zarr_meta_df.head())
print('bad zarr:', bad[:10], 'count=', len(bad))
if {'dtype', 'chunks'}.issubset(zarr_meta_df.columns):
    display(zarr_meta_df.groupby(['split', 'dtype', 'chunks']).size().rename('count').reset_index())
else:
    print('dtype/chunks columns are missing because all Zarr reads failed. Check dependency/install errors above.')

## 5. GEFF Summary Cache

In [ ]:
def read_json_safe(path):
    path = Path(path)
    if not path.exists():
        return None
    with open(path, 'r') as f:
        return json.load(f)


def find_key_recursive(obj, key):
    if isinstance(obj, dict):
        if key in obj:
            return obj[key]
        for value in obj.values():
            found = find_key_recursive(value, key)
            if found is not None:
                return found
    elif isinstance(obj, list):
        for value in obj:
            found = find_key_recursive(value, key)
            if found is not None:
                return found
    return None


def geff_sample_summary(geff_path):
    nodes_df, edges_df = bh.load_geff_annotations(geff_path)
    out_counts = edges_df['source_id'].value_counts() if len(edges_df) else pd.Series(dtype=int)
    division_nodes = out_counts[out_counts >= 2]
    nodes_per_t = nodes_df.groupby('t').size() if len(nodes_df) else pd.Series(dtype=int)
    root_meta = read_json_safe(Path(geff_path) / 'zarr.json')
    estimated = find_key_recursive(root_meta, 'estimated_number_of_nodes') if root_meta else None
    return {
        'sample_id': Path(geff_path).stem,
        'embryo_id': Path(geff_path).stem.split('_')[0],
        'n_gt_nodes': len(nodes_df),
        'n_gt_edges': len(edges_df),
        't_min': int(nodes_df['t'].min()) if len(nodes_df) else np.nan,
        't_max': int(nodes_df['t'].max()) if len(nodes_df) else np.nan,
        'gt_nodes_per_t_median': float(nodes_per_t.median()) if len(nodes_per_t) else 0.0,
        'gt_nodes_per_t_max': int(nodes_per_t.max()) if len(nodes_per_t) else 0,
        'division_nodes': int(len(division_nodes)),
        'max_out_degree': int(out_counts.max()) if len(out_counts) else 0,
        'estimated_number_of_nodes': int(estimated) if estimated is not None else np.nan,
    }


geff_summary_path = OUTPUT_DIR / 'geff_summary_199.csv'
if geff_summary_path.exists() and not FORCE_REBUILD_GEFF_SUMMARY:
    geff_summary_df = pd.read_csv(geff_summary_path)
    print('Loaded cached:', geff_summary_path)
elif RUN_FULL_GEFF_SUMMARY:
    geff_paths = [TRAIN_DIR / f'{sample_id}.geff' for sample_id in paired_ids]
    geff_summary_df = pd.DataFrame([geff_sample_summary(p) for p in tqdm(geff_paths, desc='GEFF summary')])
    geff_summary_df.to_csv(geff_summary_path, index=False)
    print('Saved:', geff_summary_path)
else:
    raise RuntimeError('GEFF summary missing. Set RUN_FULL_GEFF_SUMMARY=True.')

display(geff_summary_df.head())
display(geff_summary_df.describe(include='all'))
display(geff_summary_df.groupby('embryo_id')[['n_gt_nodes', 'n_gt_edges', 'division_nodes']].sum().reset_index())

## 6. Division ve Yogunluk Listeleri

In [ ]:
division_df = geff_summary_df[geff_summary_df['division_nodes'] > 0].sort_values(
    ['division_nodes', 'n_gt_nodes'], ascending=False
).reset_index(drop=True)

print('samples with divisions:', len(division_df))
print('total division nodes:', int(geff_summary_df['division_nodes'].sum()))
display(division_df.head(30))

density_cols = ['sample_id', 'embryo_id', 'n_gt_nodes', 'n_gt_edges', 'division_nodes', 'estimated_number_of_nodes', 'gt_nodes_per_t_median', 'gt_nodes_per_t_max']
display(geff_summary_df[density_cols].sort_values('n_gt_nodes').head(10))
display(geff_summary_df[density_cols].sort_values('n_gt_nodes', ascending=False).head(10))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
geff_summary_df['n_gt_nodes'].hist(ax=axes[0], bins=30)
axes[0].set_title('GT nodes per sample')
geff_summary_df['division_nodes'].hist(ax=axes[1], bins=range(0, int(geff_summary_df['division_nodes'].max()) + 2))
axes[1].set_title('division nodes per sample')
geff_summary_df['estimated_number_of_nodes'].hist(ax=axes[2], bins=30)
axes[2].set_title('estimated_number_of_nodes')
plt.tight_layout()
plt.show()

## 7. Stratified Validation Set Secimi

In [ ]:
def pick_density_samples(df, embryo, n_each=2):
    sub = df[df['embryo_id'] == embryo].sort_values('n_gt_nodes').reset_index(drop=True)
    picks = []
    if len(sub) == 0:
        return picks
    quantile_positions = [0.10, 0.50, 0.90]
    for q in quantile_positions:
        center = int(round(q * (len(sub) - 1)))
        lo = max(0, center - n_each // 2)
        hi = min(len(sub), lo + n_each)
        picks.extend(sub.iloc[lo:hi]['sample_id'].tolist())
    return picks


selected = []
reasons = {}

for embryo in sorted(geff_summary_df['embryo_id'].unique()):
    for sid in pick_density_samples(geff_summary_df, embryo, n_each=2):
        selected.append(sid)
        reasons.setdefault(sid, []).append(f'{embryo}_density_quantile')

for sid in division_df.head(8)['sample_id'].tolist():
    selected.append(sid)
    reasons.setdefault(sid, []).append('top_division')

for sid in geff_summary_df.sort_values('estimated_number_of_nodes', ascending=False).head(4)['sample_id'].tolist():
    selected.append(sid)
    reasons.setdefault(sid, []).append('high_estimated_count')

validation_ids = []
for sid in selected:
    if sid not in validation_ids:
        validation_ids.append(sid)

validation_df = geff_summary_df[geff_summary_df['sample_id'].isin(validation_ids)].copy()
validation_df['selection_reason'] = validation_df['sample_id'].map(lambda sid: ','.join(reasons.get(sid, [])))
validation_df = validation_df.sort_values(['embryo_id', 'selection_reason', 'n_gt_nodes']).reset_index(drop=True)

validation_path = OUTPUT_DIR / 'validation_samples.csv'
validation_df.to_csv(validation_path, index=False)
print('validation samples:', len(validation_df))
print('saved:', validation_path)
display(validation_df[density_cols + ['selection_reason']])
display(validation_df.groupby('embryo_id').size().rename('count').reset_index())

## 8. Representative Intensity Sampling

In [ ]:
def intensity_summary_for_sample(sample_id, timepoints=INTENSITY_TIMEPOINTS):
    sample_path = TRAIN_DIR / f'{sample_id}.zarr'
    img = bh.open_image_zarr(sample_path, print_info=False)
    rows = []
    for t in timepoints:
        t = int(min(max(0, t), img.shape[0] - 1))
        vol = np.asarray(img[t])
        rows.append({
            'sample_id': sample_id,
            'embryo_id': sample_id.split('_')[0],
            't': t,
            'min': int(vol.min()),
            'p1': float(np.percentile(vol, 1)),
            'p50': float(np.percentile(vol, 50)),
            'p99': float(np.percentile(vol, 99)),
            'p99_8': float(np.percentile(vol, 99.8)),
            'max': int(vol.max()),
            'mean': float(vol.mean()),
        })
    return pd.DataFrame(rows)


if RUN_REPRESENTATIVE_INTENSITY:
    intensity_ids = validation_df['sample_id'].tolist()
    intensity_df = pd.concat([intensity_summary_for_sample(sid) for sid in tqdm(intensity_ids, desc='representative intensity')], ignore_index=True)
    intensity_path = OUTPUT_DIR / 'representative_intensity.csv'
    intensity_df.to_csv(intensity_path, index=False)
    print('saved:', intensity_path)
    display(intensity_df.head())
    display(intensity_df.groupby(['embryo_id', 'sample_id'])[['p1', 'p50', 'p99', 'p99_8', 'max', 'mean']].median().reset_index())
else:
    print('RUN_REPRESENTATIVE_INTENSITY=False')

## 9. Boundary / FOV Edge Check on Validation Samples

In [ ]:
def boundary_stats_for_sample(sample_id, margin=3):
    nodes_df, _ = bh.load_geff_annotations(TRAIN_DIR / f'{sample_id}.geff')
    if len(nodes_df) == 0:
        return {'sample_id': sample_id, 'n_nodes': 0}
    return {
        'sample_id': sample_id,
        'embryo_id': sample_id.split('_')[0],
        'n_nodes': len(nodes_df),
        'near_z_boundary_frac': float(((nodes_df['z'] <= margin) | (nodes_df['z'] >= 63 - margin)).mean()),
        'near_y_boundary_frac': float(((nodes_df['y'] <= margin) | (nodes_df['y'] >= 255 - margin)).mean()),
        'near_x_boundary_frac': float(((nodes_df['x'] <= margin) | (nodes_df['x'] >= 255 - margin)).mean()),
        't_coverage_frac': float(nodes_df['t'].nunique() / 100),
    }


if RUN_SELECTED_BOUNDARY_ANALYSIS:
    boundary_df = pd.DataFrame([boundary_stats_for_sample(sid) for sid in tqdm(validation_df['sample_id'], desc='boundary selected')])
    boundary_path = OUTPUT_DIR / 'validation_boundary_stats.csv'
    boundary_df.to_csv(boundary_path, index=False)
    print('saved:', boundary_path)
    display(boundary_df)
else:
    print('RUN_SELECTED_BOUNDARY_ANALYSIS=False')

## 10. Runtime Profiling Skeleton

In [ ]:
def profile_detection_sample(sample_id, max_timepoints=RUNTIME_PROFILE_TIMEPOINTS):
    sample_path = TRAIN_DIR / f'{sample_id}.zarr'
    img = bh.open_image_zarr(sample_path, print_info=False)
    start = time.time()
    detections = bh.detect_cells_for_sample(img, sample_id, bh.DETECTION_PARAMS, max_timepoints=max_timepoints)
    elapsed = time.time() - start
    return {
        'sample_id': sample_id,
        'max_timepoints': max_timepoints,
        'detections': len(detections),
        'seconds': elapsed,
        'seconds_per_timepoint': elapsed / max_timepoints,
        'estimated_seconds_100t': elapsed / max_timepoints * 100,
    }


if RUN_RUNTIME_PROFILE:
    profile_ids = validation_df.sort_values('n_gt_nodes').iloc[[0, len(validation_df)//2, -1]]['sample_id'].tolist()
    runtime_df = pd.DataFrame([profile_detection_sample(sid) for sid in profile_ids])
    display(runtime_df)
else:
    print('RUN_RUNTIME_PROFILE=False. Acmak icin flagi True yap.')

## 11. Visual Spot Checks

Median-z views can miss sparse GT labels. This helper picks the densest `(t, z)` slab within a z tolerance and saves the reviewed sample list.

In [ ]:
RUN_VISUAL_SPOT_CHECKS = False
VISUAL_Z_TOLERANCE = 2


def choose_densest_tz_slab(nodes_df, z_tolerance=VISUAL_Z_TOLERANCE):
    if len(nodes_df) == 0:
        return 0, 32, pd.DataFrame()

    t_counts = nodes_df.groupby('t').size()
    t = int(t_counts.idxmax())
    frame_nodes = nodes_df[nodes_df['t'] == t].copy()
    if len(frame_nodes) == 0:
        return t, 32, frame_nodes

    z_values = sorted(frame_nodes['z'].astype(int).unique())
    best_z = int(np.median(z_values))
    best_count = -1
    for z in z_values:
        count = int((frame_nodes['z'].sub(z).abs() <= z_tolerance).sum())
        if count > best_count:
            best_count = count
            best_z = int(z)

    slab_nodes = frame_nodes[frame_nodes['z'].sub(best_z).abs() <= z_tolerance].copy()
    return t, best_z, slab_nodes


def plot_densest_gt_slab(sample_id, z_tolerance=VISUAL_Z_TOLERANCE):
    img = bh.open_image_zarr(TRAIN_DIR / f'{sample_id}.zarr', print_info=False)
    nodes_df, edges_df = bh.load_geff_annotations(TRAIN_DIR / f'{sample_id}.geff')
    t, z, slab_nodes = choose_densest_tz_slab(nodes_df, z_tolerance=z_tolerance)
    volume = np.asarray(img[t])
    plane = volume[z]

    meta = geff_summary_df.set_index('sample_id').loc[sample_id]
    plt.figure(figsize=(6, 6))
    plt.imshow(plane, cmap='gray', vmin=np.percentile(plane, 1), vmax=np.percentile(plane, 99.8))
    if len(slab_nodes):
        plt.scatter(slab_nodes['x'], slab_nodes['y'], s=18, facecolors='none', edgecolors='red', linewidths=1)
    plt.title(
        f'{sample_id} | t={t}, z={z}, slab_nodes={len(slab_nodes)}\n'
        f'GT nodes={int(meta.n_gt_nodes)}, divisions={int(meta.division_nodes)}, est={int(meta.estimated_number_of_nodes)}'
    )
    plt.axis('off')
    plt.show()


def build_visual_spotcheck_ids(validation_df):
    preferred = [
        '44b6_0c582fdc', '6bba_afb141ff', '6bba_09961292', '6bba_48816121',
        '6bba_df673a83', '6bba_cdcfe533', '44b6_d754aa59', '6bba_3db54e20',
        '6bba_c328f2fd',
    ]
    available = set(validation_df['sample_id'])
    ids = [sid for sid in preferred if sid in available]

    density_ids = validation_df.sort_values('n_gt_nodes').iloc[[0, len(validation_df)//2, -1]]['sample_id'].tolist()
    division_ids = validation_df.sort_values(['division_nodes', 'n_gt_nodes'], ascending=False).head(4)['sample_id'].tolist()
    for sid in density_ids + division_ids:
        if sid not in ids:
            ids.append(sid)
    return ids


visual_ids = build_visual_spotcheck_ids(validation_df)
visual_ids_path = OUTPUT_DIR / 'visual_spotcheck_ids.csv'
pd.DataFrame({'sample_id': visual_ids}).to_csv(visual_ids_path, index=False)
print('visual spot check sample ids:', visual_ids)
print('saved:', visual_ids_path)

if RUN_VISUAL_SPOT_CHECKS:
    for sample_id in visual_ids:
        plot_densest_gt_slab(sample_id)
else:
    print('RUN_VISUAL_SPOT_CHECKS=False. Acmak icin flagi True yap.')

## 12. Sonraki Adimlar

- `validation_samples.csv` uzerinden local scorer ve parameter sweep kur.
- Detection sweep: `percentile_high`, `threshold_abs`, `gaussian_sigma`, `min_distance`.
- Linking sweep: `max_distance_um = 5,6,7,8,9,10`.
- Division samples uzerinde binary split candidate logic gelistir.
- Final submission notebook sadece secilen pipeline'i calistirsin; EDA tasimasin.